## 依赖导入

博弈论一节：矩阵对策的均衡条件用 numpy 构造增广线性方程组、sympy 精确行最简形与伪逆最小范数解求解；对策值用 cvxpy 的一对线性规划（LP）计算；连续对策的混合平衡用 scipy.optimize.minimize 的非线性约束优化求解。本例无绘图，不需要 matplotlib 全局配置。

In [ ]:
import cvxpy as cp
import numpy as np
import sympy as sp
from scipy.optimize import minimize

## 14.3  矩阵对策的均衡条件与线性方程组

6×6 实对策矩阵 A（每行一个策略）。把局中一的均衡条件写成 $A^{\top}x-u=0$：`Az1` 是 $A^{\top}$ 横接 -1 列，`Az2` 再纵接归一化方程 [1,…,1 | 0]，构成完整的系数阵；`B` 为常数项列（末行 1、其余 0），横接后得增广阵 `Az3`。`sp.Matrix(Az3.astype(int)).rref()` 化为精确行最简形（打印 s1[0]），`np.linalg.pinv(Az2) @ B` 求最小范数解（打印 s2）。最后 `np.savetxt("data/16_3.txt", A, fmt="%.0f")` 保存对策矩阵，供 14.4 读取（源程序 ex16_3.py）。

In [ ]:
A = np.array(
    [
        [3, 1, 1, 1, 1, -1],
        [1, 3, 1, 1, -1, 1],
        [1, -1, 3, 1, 1, 1],
        [-1, 1, 1, 3, 1, 1],
        [1, 1, -1, 1, 3, 1],
        [1, 1, 1, -1, 1, 3],
    ],
    dtype=int,
)
Az1 = np.hstack([A.T, -np.ones((6, 1))])
Az2 = np.vstack([Az1, [1, 1, 1, 1, 1, 1, 0]])
B = np.array([[0, 0, 0, 0, 0, 0, 1]]).T
Az3 = np.hstack([Az2, B])
Az4 = sp.Matrix(Az3.astype(int))
s1 = Az4.rref()
s2 = np.linalg.pinv(Az2) @ B
print("行最简形为：\n", s1[0])
print("最小范数解为：\n", s2)
np.savetxt("data/16_3.txt", A, fmt="%.0f")

## 14.4  两个线性规划求解矩阵对策（数据来自 14.3）

读 `data/16_3.txt` 的支付矩阵 A，变量 x、y 均 6 维且 pos=True（非负）。局中一 LP：变量 u，Maximize u，约束 $A^{\top}x\ge u$、$\sum x=1$；局中二 LP：变量 v，Minimize v，约束 $Ay\le v$、$\sum y=1$（约束中的 sum 作用于 cvxpy 表达式，保持原写法）。两个 LP 均以 GLPK_MI 求解，分别打印最优值 u、最优解 x 与最优值 v、最优解 y（源程序 ex16_4.py）。最优时 u=v 即对策值。

In [ ]:
A = np.loadtxt("data/16_3.txt")
x = cp.Variable(6, pos=True)
y = cp.Variable(6, pos=True)
u = cp.Variable()
v = cp.Variable()
ob1 = cp.Maximize(u)
con1 = [A.T @ x >= u, sum(x) == 1]
prob1 = cp.Problem(ob1, con1)
prob1.solve(solver="GLPK_MI")
print("最优值 u:", prob1.value)
print("最优解 x:\n", x.value)
ob2 = cp.Minimize(v)
con2 = [A @ y <= v, sum(y) == 1]
prob2 = cp.Problem(ob2, con2)
prob2.solve(solver="GLPK_MI")
print("最优解 v:", prob2.value)
print("最优解 y:\n", y.value)

## 14.5  3×3 具体对策的双 LP 解法

给定 3×3 支付矩阵（元素为 1/3、1/2、-2/5 等分数），方法与 14.4 完全相同：构造 Maximize u（$A^{\top}x\ge u$、$\sum x=1$）与 Minimize v（$Ay\le v$、$\sum y=1$）一对线性规划，GLPK_MI 求解，打印最优值与 round 4 位的最优解 x、y（源程序 ex16_5.py）。

In [ ]:
A = np.array([[1 / 3, 1 / 2, -1 / 3], [-2 / 5, 1 / 5, -1 / 2], [1 / 2, -3 / 5, 1 / 3]])
x = cp.Variable(3, pos=True)
y = cp.Variable(3, pos=True)
u = cp.Variable()
v = cp.Variable()
ob1 = cp.Maximize(u)
con1 = [A.T @ x >= u, sum(x) == 1]
prob1 = cp.Problem(ob1, con1)
prob1.solve(solver="GLPK_MI")
print("最优值 u:", prob1.value)
print("最优解 x:\n", np.round(x.value, 4))
ob2 = cp.Minimize(v)
con2 = [A @ y <= v, sum(y) == 1]
prob2 = cp.Problem(ob2, con2)
prob2.solve(solver="GLPK_MI")
print("最优解 v:", prob2.value)
print("最优解 y:\n", np.round(y.value, 4))

## 14.7  连续对策混合平衡的非线性规划解法

甲、乙两队的 3×3 支付矩阵 a、b。决策向量 z 的前三分量是甲的混合策略 x、后三个是乙的混合策略 y。用 `scipy.optimize.minimize`（SLSQP）解平衡：虚拟目标 `obj(z)` 取 z 的分量和（n=6，`.sum()` 与内置 sum 逐次同序累加、逐位一致）；不等式约束 $x^{\top}a\ge ay$（`z[:3] @ a @ z[3:] - a @ z[3:]` 的每个分量非负）与 $x^{\top}b\ge b^{\top}x$；等式约束 x、y 两段概率和为 1；bounds 各分量 (0,1)。打印优化结果 s 的详细信息、x、y 及两队平均得分 x·a·y、x·b·y（源程序 ex16_7.py）。

In [ ]:
a = np.array([[14, 13, 12], [13, 12, 12], [12, 12, 13]])
b = np.array([[13, 14, 15], [14, 15, 15], [15, 15, 14]])


def obj(z):
    return z.sum()


con1 = {"type": "ineq", "fun": lambda z: z[:3] @ a @ z[3:] - a @ z[3:]}
con2 = {"type": "ineq", "fun": lambda z: z[:3] @ b @ z[3:] - b.T @ z[:3]}
con3 = {"type": "eq", "fun": lambda z: z[:3].sum() - 1}
con4 = {"type": "eq", "fun": lambda z: z[3:].sum() - 1}
con = [con1, con2, con3, con4]
bd = [(0, 1) for i in range(6)]
s = minimize(obj, np.ones(6), constraints=con, bounds=bd)
print("解的详细信息如下：\n", s)
x = s.x[:3]
y = s.x[3:]
print("x 的解为:", x)
print("y 的解为:", y)
print("甲队平均得分:", x @ a @ y)
print("乙队平均得分:", x @ b @ y)